In [ ]:
import os
import sqlite3
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox as msgbox

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
from matplotlib.ticker import ScalarFormatter


# ============================================================
# 基本設定
# ============================================================
DB_PATH = r"D:\database\car.db"

plt.rcParams["font.family"] = ["Microsoft JhengHei"]
plt.rcParams["axes.unicode_minus"] = False

COLORS = {
    "bg": "#F3F6FB",
    "surface": "#FFFFFF",
    "surface_alt": "#F8FAFC",
    "navy": "#111827",
    "primary": "#2563EB",
    "primary_hover": "#1D4ED8",
    "violet": "#7C3AED",
    "green": "#059669",
    "orange": "#EA580C",
    "text": "#111827",
    "muted": "#64748B",
    "border": "#E2E8F0",
    "danger": "#DC2626",
}

col_map = {
    "車型": "brand",
    "型號": "model",
    "價格": "price_usd",
    "馬力": "horsepower",
    "最高時速": "top_speed",
    "稀有度": "rarity",
    "外觀": "design_rating",
    "推出年分": "release_year",
}

ATTRS = ["價格", "馬力", "最高時速", "外觀", "推出年分"]

# 全域變數（保留原本程式的操作方式）
combo_brand = combo_x = combo_y = None
combo1 = combo2 = combo3 = None
comboa = combob = comboc = None
brand_combo = model_combo = tree = None
canvas_frame = None
p1 = None
fig = None
fig_win = None
win_1 = None
win_4 = None
prediction_model_ready = False
a = b = None

B = M = P = H = T = R = D = Y = None


# ============================================================
# 共用 UI 工具
# ============================================================
def center_window(window, width, height):
    """讓視窗在螢幕中央開啟。"""
    window.update_idletasks()
    sw = window.winfo_screenwidth()
    sh = window.winfo_screenheight()
    x = max((sw - width) // 2, 0)
    y = max((sh - height) // 2, 0)
    window.geometry(f"{width}x{height}+{x}+{y}")


def setup_style():
    style = ttk.Style()
    try:
        style.theme_use("clam")
    except tk.TclError:
        pass

    style.configure(
        "TCombobox",
        font=("Microsoft JhengHei", 11),
        padding=7,
        fieldbackground=COLORS["surface"],
        background=COLORS["surface"],
        bordercolor=COLORS["border"],
        lightcolor=COLORS["border"],
        darkcolor=COLORS["border"],
        arrowsize=14,
    )

    style.configure(
        "Treeview",
        font=("Microsoft JhengHei", 10),
        rowheight=34,
        background=COLORS["surface"],
        fieldbackground=COLORS["surface"],
        foreground=COLORS["text"],
        borderwidth=0,
    )
    style.configure(
        "Treeview.Heading",
        font=("Microsoft JhengHei", 10, "bold"),
        background=COLORS["surface_alt"],
        foreground=COLORS["text"],
        relief="flat",
        padding=8,
    )
    style.map("Treeview", background=[("selected", "#DBEAFE")], foreground=[("selected", COLORS["text"])])

    style.configure(
        "Primary.TButton",
        font=("Microsoft JhengHei", 10, "bold"),
        padding=(16, 9),
        background=COLORS["primary"],
        foreground="white",
        borderwidth=0,
    )
    style.map(
        "Primary.TButton",
        background=[("active", COLORS["primary_hover"]), ("pressed", COLORS["primary_hover"])],
        foreground=[("active", "white")],
    )

    style.configure(
        "Secondary.TButton",
        font=("Microsoft JhengHei", 10, "bold"),
        padding=(16, 9),
        background=COLORS["surface_alt"],
        foreground=COLORS["text"],
        bordercolor=COLORS["border"],
        borderwidth=1,
    )
    style.map("Secondary.TButton", background=[("active", "#EEF2FF")])

    style.configure(
        "Success.TButton",
        font=("Microsoft JhengHei", 10, "bold"),
        padding=(16, 9),
        background=COLORS["green"],
        foreground="white",
        borderwidth=0,
    )
    style.map("Success.TButton", background=[("active", "#047857")])


def make_label(parent, text, size=11, weight="normal", color=None, bg=None, **kwargs):
    return tk.Label(
        parent,
        text=text,
        font=("Microsoft JhengHei", size, weight),
        fg=color or COLORS["text"],
        bg=bg or parent.cget("bg"),
        **kwargs,
    )


def make_entry(parent, variable, width=22):
    return tk.Entry(
        parent,
        textvariable=variable,
        width=width,
        font=("Microsoft JhengHei", 11),
        relief="flat",
        bg=COLORS["surface_alt"],
        fg=COLORS["text"],
        insertbackground=COLORS["text"],
        highlightthickness=1,
        highlightbackground=COLORS["border"],
        highlightcolor=COLORS["primary"],
    )


def make_card(parent, padx=20, pady=18):
    outer = tk.Frame(parent, bg=COLORS["border"])
    inner = tk.Frame(outer, bg=COLORS["surface"])
    inner.pack(fill="both", expand=True, padx=1, pady=1)
    inner.configure(padx=padx, pady=pady)
    return outer, inner


def add_window_header(parent, title, subtitle):
    header = tk.Frame(parent, bg=COLORS["navy"], height=92)
    header.pack(fill="x")
    header.pack_propagate(False)

    text = tk.Frame(header, bg=COLORS["navy"])
    text.pack(side="left", padx=28, pady=18)
    make_label(text, title, 20, "bold", "white", COLORS["navy"]).pack(anchor="w")
    make_label(text, subtitle, 10, "normal", "#CBD5E1", COLORS["navy"]).pack(anchor="w", pady=(3, 0))
    return header


def safe_get_brands():
    try:
        return get_brands()
    except Exception as exc:
        msgbox.showerror("資料庫錯誤", f"無法讀取資料庫：\n{DB_PATH}\n\n{exc}")
        return []


def show_db_error(exc):
    msgbox.showerror("資料庫錯誤", f"無法操作資料庫：\n{DB_PATH}\n\n{exc}")


def draw_empty_chart(parent, text="請先選擇條件並生成圖表"):
    for widget in parent.winfo_children():
        widget.destroy()
    holder = tk.Frame(parent, bg=COLORS["surface_alt"])
    holder.pack(fill="both", expand=True)
    make_label(holder, "CHART", 11, "bold", COLORS["primary"], COLORS["surface_alt"]).pack(pady=(90, 8))
    make_label(holder, text, 13, "bold", COLORS["muted"], COLORS["surface_alt"]).pack()


# ============================================================
# 資料庫
# ============================================================
def get_brands():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT DISTINCT brand FROM supercars ORDER BY brand")
    brands = [row[0] for row in cursor.fetchall()]
    conn.close()
    return brands


# ============================================================
# 屬性趨勢預測
# ============================================================
def predict_menu():
    global combo_brand, combo_x, combo_y, p1, canvas_frame, prediction_model_ready

    prediction_model_ready = False
    pr_win = tk.Toplevel(win)
    pr_win.title("車輛屬性趨勢預測")
    pr_win.configure(bg=COLORS["bg"])
    center_window(pr_win, 980, 760)
    pr_win.minsize(900, 680)

    add_window_header(
        pr_win,
        "車輛屬性趨勢預測",
        "選擇品牌與兩個屬性，建立線性回歸模型並輸入數值進行預測。",
    )

    body = tk.Frame(pr_win, bg=COLORS["bg"])
    body.pack(fill="both", expand=True, padx=24, pady=22)

    control_outer, control = make_card(body, 20, 16)
    control_outer.pack(fill="x")

    brands = safe_get_brands()
    p1 = tk.DoubleVar()

    make_label(control, "模型條件", 13, "bold").grid(row=0, column=0, columnspan=6, sticky="w", pady=(0, 12))

    make_label(control, "品牌", 10, "bold", COLORS["muted"]).grid(row=1, column=0, sticky="w")
    combo_brand = ttk.Combobox(control, values=brands, state="readonly", width=18)
    combo_brand.grid(row=2, column=0, padx=(0, 16), pady=(5, 0), sticky="ew")
    if brands:
        combo_brand.set(brands[0])

    make_label(control, "X 軸屬性", 10, "bold", COLORS["muted"]).grid(row=1, column=1, sticky="w")
    combo_x = ttk.Combobox(control, values=ATTRS, state="readonly", width=16)
    combo_x.grid(row=2, column=1, padx=(0, 16), pady=(5, 0), sticky="ew")
    combo_x.set("價格")

    make_label(control, "Y 軸屬性", 10, "bold", COLORS["muted"]).grid(row=1, column=2, sticky="w")
    combo_y = ttk.Combobox(control, values=ATTRS, state="readonly", width=16)
    combo_y.grid(row=2, column=2, padx=(0, 16), pady=(5, 0), sticky="ew")
    combo_y.set("馬力")

    ttk.Button(control, text="生成預測圖", command=predict_car, style="Primary.TButton").grid(
        row=2, column=3, padx=(8, 0), sticky="ew"
    )
    control.grid_columnconfigure(0, weight=1)
    control.grid_columnconfigure(1, weight=1)
    control.grid_columnconfigure(2, weight=1)

    chart_outer, chart_card = make_card(body, 12, 12)
    chart_outer.pack(fill="both", expand=True, pady=(18, 0))
    canvas_frame = tk.Frame(chart_card, bg=COLORS["surface"])
    canvas_frame.pack(fill="both", expand=True)
    draw_empty_chart(canvas_frame)

    predict_outer, predict_card = make_card(body, 20, 14)
    predict_outer.pack(fill="x", pady=(18, 0))

    make_label(predict_card, "用 X 值預測 Y 值", 12, "bold").grid(row=0, column=0, sticky="w", padx=(0, 16))
    make_label(predict_card, "輸入 X 軸數值", 10, "normal", COLORS["muted"]).grid(row=0, column=1, sticky="e", padx=(0, 8))
    input_entry = make_entry(predict_card, p1, 16)
    input_entry.grid(row=0, column=2, sticky="ew", padx=(0, 12), ipady=6)
    ttk.Button(predict_card, text="開始預測", command=start_car_predict, style="Success.TButton").grid(row=0, column=3)
    predict_card.grid_columnconfigure(2, weight=1)


def predict_car():
    global a, b, x_vals, y_vals, x_col, y_col, prediction_model_ready

    brand = combo_brand.get()
    if not brand:
        msgbox.showwarning("提醒", "請先選擇品牌。")
        return
    if combo_x.get() == combo_y.get():
        msgbox.showwarning("提醒", "X 軸與 Y 軸請選擇不同屬性。")
        return

    x_col = col_map[combo_x.get()]
    y_col = col_map[combo_y.get()]

    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql_query(
            f"SELECT {x_col}, {y_col} FROM supercars WHERE brand = ? ORDER BY {x_col}",
            conn,
            params=(brand,),
        )
        conn.close()
    except Exception as exc:
        show_db_error(exc)
        return

    df = df.dropna()
    if len(df) < 2:
        msgbox.showwarning("資料不足", "此品牌的有效資料不足，無法建立模型。")
        return

    x_vals = np.array(df[x_col], dtype=float)
    y_vals = np.array(df[y_col], dtype=float)
    x_mean = np.mean(x_vals)
    y_mean = np.mean(y_vals)
    numerator = np.sum((x_vals - x_mean) * (y_vals - y_mean))
    denominator = np.sum((x_vals - x_mean) ** 2)

    if denominator == 0:
        msgbox.showwarning("資料不足", "X 軸資料沒有變化，無法建立線性回歸模型。")
        return

    a = numerator / denominator
    b = y_mean - a * x_mean
    prediction_model_ready = True

    order = np.argsort(x_vals)
    x_sorted = x_vals[order]

    chart_fig = Figure(figsize=(7.8, 4.4), dpi=100, facecolor=COLORS["surface"])
    ax = chart_fig.add_subplot(111)
    ax.scatter(x_vals, y_vals, s=45, alpha=0.78, label="實際數據")
    ax.plot(x_sorted, a * x_sorted + b, linewidth=2.2, label="線性回歸線")
    ax.ticklabel_format(style="plain", axis="both")

    x_label = "價格 (USD)" if combo_x.get() == "價格" else combo_x.get()
    y_label = "價格 (USD)" if combo_y.get() == "價格" else combo_y.get()
    ax.set_xlabel(x_label, fontsize=10)
    ax.set_ylabel(y_label, fontsize=10)
    ax.set_title(f"{brand}｜{combo_x.get()} × {combo_y.get()}", fontsize=15, fontweight="bold", pad=12)
    ax.grid(True, alpha=0.2)
    ax.legend(frameon=False)
    chart_fig.tight_layout()

    for widget in canvas_frame.winfo_children():
        widget.destroy()
    canvas = FigureCanvasTkAgg(chart_fig, master=canvas_frame)
    canvas.draw()
    canvas.get_tk_widget().pack(fill="both", expand=True)


def start_car_predict():
    if not prediction_model_ready or a is None or b is None:
        msgbox.showwarning("提醒", "請先按「生成預測圖」建立模型。")
        return

    try:
        x_input = float(p1.get())
    except Exception:
        msgbox.showwarning("提醒", "請輸入有效的數值。")
        return

    y_pred = a * x_input + b
    msgbox.showinfo(
        "預測結果",
        f"當 {combo_x.get()} = {x_input:,.2f} 時\n預測 {combo_y.get()} ≈ {y_pred:,.2f}",
    )


# ============================================================
# 圖表視窗
# ============================================================
def create_relation_window(title, subtitle, button_command):
    chart_win = tk.Toplevel(win)
    chart_win.title(title)
    chart_win.configure(bg=COLORS["bg"])
    center_window(chart_win, 900, 690)
    chart_win.minsize(800, 600)

    add_window_header(chart_win, title, subtitle)

    body = tk.Frame(chart_win, bg=COLORS["bg"])
    body.pack(fill="both", expand=True, padx=24, pady=22)

    filter_outer, filter_card = make_card(body, 20, 14)
    filter_outer.pack(fill="x")

    brands = safe_get_brands()
    brand_cb = ttk.Combobox(filter_card, values=brands, state="readonly", width=24)
    make_label(filter_card, "選擇品牌", 10, "bold", COLORS["muted"]).pack(side="left", padx=(0, 12))
    brand_cb.pack(side="left", padx=(0, 14))
    if brands:
        default = "Lamborghini" if "Lamborghini" in brands else brands[0]
        brand_cb.set(default)

    chart_holder_outer, chart_holder = make_card(body, 12, 12)
    chart_holder_outer.pack(fill="both", expand=True, pady=(18, 0))

    btn = ttk.Button(filter_card, text="生成圖表", style="Primary.TButton")
    btn.pack(side="left")
    btn.configure(command=lambda: button_command(brand_cb.get(), chart_holder))

    draw_empty_chart(chart_holder)
    return chart_win, brand_cb, chart_holder


def render_line_chart(holder, x_values, y_values, title, xlabel, ylabel, label):
    for widget in holder.winfo_children():
        widget.destroy()

    chart_fig = Figure(figsize=(7.6, 4.7), dpi=100, facecolor=COLORS["surface"])
    ax = chart_fig.add_subplot(111)
    ax.plot(x_values, y_values, marker="o", linewidth=2.1, markersize=5, label=label)
    ax.set_title(title, fontsize=16, fontweight="bold", pad=12)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.ticklabel_format(style="plain", axis="both")
    ax.grid(True, alpha=0.2)
    ax.legend(frameon=False)
    chart_fig.tight_layout()

    canvas = FigureCanvasTkAgg(chart_fig, master=holder)
    canvas.draw()
    canvas.get_tk_widget().pack(fill="both", expand=True)


def money():
    create_relation_window(
        "車價歷年趨勢",
        "依品牌查看不同推出年份的車款價格變化。",
        render_price_trend,
    )


def render_price_trend(brand, holder):
    if not brand:
        msgbox.showwarning("提醒", "請先選擇品牌。")
        return
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(
            "SELECT release_year, price_usd FROM supercars WHERE brand = ? ORDER BY release_year",
            (brand,),
        )
        rows = cursor.fetchall()
        conn.close()
    except Exception as exc:
        show_db_error(exc)
        return

    if not rows:
        msgbox.showwarning("查無資料", "這個品牌目前沒有可繪製的資料。")
        return

    years = [r[0] for r in rows]
    prices = [r[1] for r in rows]
    render_line_chart(holder, years, prices, brand, "推出年份", "價格 (USD)", f"{brand} 價格趨勢")


def find():
    """保留原函式名稱，相容舊程式操作。"""
    if combo1 is not None and fig_win is not None:
        render_price_trend(combo1.get(), fig_win)


def appearance():
    create_relation_window(
        "價格與外觀評分關係",
        "比較同品牌車款的市場價格與外觀評分變化。",
        lambda brand, holder: render_attribute_relation(brand, holder, "design_rating", "外觀評分"),
    )


def horse():
    create_relation_window(
        "價格與馬力關係",
        "比較同品牌車款的市場價格與馬力變化。",
        lambda brand, holder: render_attribute_relation(brand, holder, "horsepower", "馬力"),
    )


def render_attribute_relation(brand, holder, attr_col, attr_label):
    if not brand:
        msgbox.showwarning("提醒", "請先選擇品牌。")
        return
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(
            f"SELECT price_usd, {attr_col} FROM supercars WHERE brand = ? ORDER BY price_usd",
            (brand,),
        )
        rows = cursor.fetchall()
        conn.close()
    except Exception as exc:
        show_db_error(exc)
        return

    rows = [r for r in rows if r[0] is not None and r[1] is not None]
    if not rows:
        msgbox.showwarning("查無資料", "這個品牌目前沒有可繪製的資料。")
        return

    prices = [r[0] for r in rows]
    attrs = [r[1] for r in rows]
    render_line_chart(holder, prices, attrs, brand, "價格 (USD)", attr_label, f"{brand} 趨勢")


def find2(A, code, C, win):
    """保留舊函式名稱，若其他程式仍呼叫它也能運作。"""
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(code, (C,))
        rows = cursor.fetchall()
        conn.close()
    except Exception as exc:
        show_db_error(exc)
        return

    if not rows:
        msgbox.showwarning("查無資料", "沒有可繪製的資料。")
        return

    attrs = [row[0] for row in rows]
    prices = [row[1] for row in rows]
    if A == "appearance":
        ylabel = "外觀評分"
    elif A == "horsepower":
        ylabel = "馬力"
    else:
        ylabel = A

    # 若傳入的是一般 Frame 就直接畫；若是 Toplevel 則建立一個容器
    holder = win
    if isinstance(win, tk.Toplevel):
        for child in win.winfo_children():
            if getattr(child, "_legacy_chart_holder", False):
                holder = child
                break
        else:
            holder = tk.Frame(win, bg=COLORS["surface"])
            holder._legacy_chart_holder = True
            holder.pack(fill="both", expand=True, padx=18, pady=18)
    render_line_chart(holder, prices, attrs, C, "價格 (USD)", ylabel, f"{C} 趨勢")


# ============================================================
# 3D 視覺化
# ============================================================
def datafor3D():
    global comboa, combob, comboc

    w_2 = tk.Toplevel(win)
    w_2.title("3D 關係設定")
    w_2.configure(bg=COLORS["bg"])
    center_window(w_2, 720, 330)
    w_2.resizable(False, False)

    add_window_header(w_2, "3D 關係設定", "選擇三個屬性，建立全部超跑資料的三維散佈圖。")

    body = tk.Frame(w_2, bg=COLORS["bg"])
    body.pack(fill="both", expand=True, padx=24, pady=22)

    outer, card = make_card(body, 20, 18)
    outer.pack(fill="both", expand=True)

    labels = ["X 軸", "Y 軸", "Z 軸"]
    defaults = ["價格", "馬力", "外觀"]
    combos = []

    for i, (label, default) in enumerate(zip(labels, defaults)):
        make_label(card, label, 10, "bold", COLORS["muted"]).grid(row=0, column=i, sticky="w", padx=(0, 12))
        cb = ttk.Combobox(card, values=ATTRS, state="readonly", width=18)
        cb.grid(row=1, column=i, sticky="ew", padx=(0, 12), pady=(6, 18))
        cb.set(default)
        combos.append(cb)
        card.grid_columnconfigure(i, weight=1)

    comboa, combob, comboc = combos
    ttk.Button(card, text="建立 3D 圖", command=bc3D, style="Primary.TButton").grid(
        row=2, column=0, columnspan=3, sticky="ew"
    )


def bc3D():
    x_name = comboa.get()
    y_name = combob.get()
    z_name = comboc.get()

    if len({x_name, y_name, z_name}) < 3:
        msgbox.showwarning("提醒", "X、Y、Z 三個軸請選擇不同屬性。")
        return

    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql_query(
            f"SELECT {col_map[x_name]}, {col_map[y_name]}, {col_map[z_name]} FROM supercars",
            conn,
        )
        conn.close()
    except Exception as exc:
        show_db_error(exc)
        return

    df = df.dropna()
    if df.empty:
        msgbox.showwarning("查無資料", "沒有足夠的資料建立 3D 圖。")
        return

    X = df[col_map[x_name]].values
    Yv = df[col_map[y_name]].values
    Z = df[col_map[z_name]].values

    w_3 = tk.Toplevel(win)
    w_3.title("3D 視覺化")
    w_3.configure(bg=COLORS["bg"])
    center_window(w_3, 980, 760)

    add_window_header(w_3, "3D 資料關係", f"{x_name} × {y_name} × {z_name}")

    chart_outer, chart_card = make_card(w_3, 12, 12)
    chart_outer.pack(fill="both", expand=True, padx=24, pady=22)

    chart_fig = Figure(figsize=(9, 6), dpi=100, facecolor=COLORS["surface"])
    ax = chart_fig.add_subplot(111, projection="3d")
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        formatter = ScalarFormatter()
        formatter.set_scientific(False)
        axis.set_major_formatter(formatter)

    color_values = np.linspace(0, 1, len(X))
    ax.scatter(X, Yv, Z, c=color_values, cmap="viridis", alpha=0.78, s=35)
    ax.set_xlabel(x_name, fontsize=10, labelpad=8)
    ax.set_ylabel(y_name, fontsize=10, labelpad=8)
    ax.set_zlabel(z_name, fontsize=10, labelpad=8)
    ax.set_title(f"{x_name}－{y_name}－{z_name} 的 3D 關係", fontsize=15, fontweight="bold", pad=16)
    chart_fig.tight_layout()

    canvas = FigureCanvasTkAgg(chart_fig, master=chart_card)
    canvas.draw()
    canvas.get_tk_widget().pack(fill="both", expand=True)


# ============================================================
# 車輛查詢
# ============================================================
def serch():
    global brand_combo, model_combo, tree

    win_5 = tk.Toplevel(win)
    win_5.title("品牌車型查詢")
    win_5.configure(bg=COLORS["bg"])
    center_window(win_5, 1120, 470)
    win_5.minsize(980, 430)

    add_window_header(win_5, "品牌車型查詢", "選擇品牌與型號，快速查看價格、性能、稀有度與年份資料。")

    body = tk.Frame(win_5, bg=COLORS["bg"])
    body.pack(fill="both", expand=True, padx=24, pady=22)

    filter_outer, filter_card = make_card(body, 20, 16)
    filter_outer.pack(fill="x")

    brands = safe_get_brands()

    make_label(filter_card, "品牌", 10, "bold", COLORS["muted"]).grid(row=0, column=0, sticky="w")
    brand_combo = ttk.Combobox(filter_card, values=brands, state="readonly", width=24)
    brand_combo.grid(row=1, column=0, sticky="ew", padx=(0, 18), pady=(5, 0))
    if brands:
        brand_combo.set(brands[0])
    brand_combo.bind("<<ComboboxSelected>>", update_models)

    make_label(filter_card, "車型", 10, "bold", COLORS["muted"]).grid(row=0, column=1, sticky="w")
    model_combo = ttk.Combobox(filter_card, values=[], state="readonly", width=28)
    model_combo.grid(row=1, column=1, sticky="ew", padx=(0, 18), pady=(5, 0))

    ttk.Button(filter_card, text="查詢資料", command=show_car_info, style="Primary.TButton").grid(
        row=1, column=2, sticky="ew", pady=(5, 0)
    )
    filter_card.grid_columnconfigure(0, weight=1)
    filter_card.grid_columnconfigure(1, weight=1)

    table_outer, table_card = make_card(body, 10, 10)
    table_outer.pack(fill="both", expand=True, pady=(18, 0))

    columns = ("品牌", "車型", "價格", "馬力", "最高時速", "稀有度", "外觀", "推出年分")
    tree = ttk.Treeview(table_card, columns=columns, show="headings", height=5)
    widths = {
        "品牌": 125,
        "車型": 170,
        "價格": 130,
        "馬力": 90,
        "最高時速": 105,
        "稀有度": 95,
        "外觀": 80,
        "推出年分": 100,
    }
    for col in columns:
        tree.heading(col, text=col)
        tree.column(col, width=widths[col], minwidth=70, anchor="center")

    xbar = ttk.Scrollbar(table_card, orient="horizontal", command=tree.xview)
    tree.configure(xscrollcommand=xbar.set)
    tree.pack(fill="both", expand=True)
    xbar.pack(fill="x")

    update_models()


def update_models(event=None):
    if brand_combo is None or model_combo is None:
        return
    brand = brand_combo.get()
    if not brand:
        model_combo["values"] = []
        model_combo.set("")
        return

    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute("SELECT model FROM supercars WHERE brand = ? ORDER BY model", (brand,))
        models = [row[0] for row in cursor.fetchall()]
        conn.close()
    except Exception as exc:
        show_db_error(exc)
        return

    model_combo["values"] = models
    model_combo.set(models[0] if models else "")


def show_car_info():
    for item in tree.get_children():
        tree.delete(item)

    brand = brand_combo.get()
    model = model_combo.get()
    if not brand or not model:
        msgbox.showwarning("提醒", "請先選擇品牌與車型。")
        return

    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(
            """
            SELECT brand, model, price_usd, horsepower, top_speed,
                   rarity, design_rating, release_year
            FROM supercars
            WHERE brand = ? AND model = ?
            """,
            (brand, model),
        )
        car_data = cursor.fetchone()
        conn.close()
    except Exception as exc:
        show_db_error(exc)
        return

    if car_data:
        display = list(car_data)
        if display[2] is not None:
            try:
                display[2] = f"${float(display[2]):,.0f}"
            except Exception:
                pass
        tree.insert("", "end", values=display)
    else:
        msgbox.showinfo("查詢結果", "查無符合條件的車輛資料。")


# ============================================================
# 新增資料
# ============================================================
def input_():
    global B, M, P, H, T, R, D, Y

    w_5 = tk.Toplevel(win)
    w_5.title("新增車輛資料")
    w_5.configure(bg=COLORS["bg"])
    center_window(w_5, 780, 560)
    w_5.resizable(False, False)

    add_window_header(w_5, "新增車輛資料", "將新的超跑規格加入 car.db 的 supercars 資料表。")

    B = tk.StringVar()
    M = tk.StringVar()
    P = tk.DoubleVar()
    H = tk.IntVar()
    T = tk.DoubleVar()
    R = tk.StringVar()
    D = tk.DoubleVar()
    Y = tk.IntVar()

    body = tk.Frame(w_5, bg=COLORS["bg"])
    body.pack(fill="both", expand=True, padx=24, pady=22)

    outer, card = make_card(body, 24, 22)
    outer.pack(fill="both", expand=True)

    make_label(card, "基本資料", 13, "bold").grid(row=0, column=0, columnspan=2, sticky="w", pady=(0, 16))

    fields = [
        ("品牌", B, "例如 Porsche"),
        ("車型", M, "例如 918 Spyder"),
        ("價格 (USD)", P, "例如 845000"),
        ("馬力 (hp)", H, "例如 887"),
        ("最高時速 (km/h)", T, "例如 345"),
        ("稀有度", R, "例如 Rare / 9"),
        ("外觀評分", D, "例如 9.5"),
        ("推出年分", Y, "例如 2013"),
    ]

    entries = []
    for i, (label, var, hint) in enumerate(fields):
        row = i // 2 + 1
        col = (i % 2) * 2
        make_label(card, label, 10, "bold", COLORS["muted"]).grid(
            row=row * 2 - 1, column=col, sticky="w", padx=(0, 16), pady=(0, 5)
        )
        entry = make_entry(card, var, 26)
        entry.grid(row=row * 2, column=col, columnspan=2, sticky="ew", padx=(0, 18), pady=(0, 14), ipady=7)
        entries.append(entry)

    card.grid_columnconfigure(0, weight=1)
    card.grid_columnconfigure(1, weight=1)
    card.grid_columnconfigure(2, weight=1)
    card.grid_columnconfigure(3, weight=1)

    actions = tk.Frame(card, bg=COLORS["surface"])
    actions.grid(row=10, column=0, columnspan=4, sticky="ew", pady=(8, 0))
    ttk.Button(actions, text="取消", command=w_5.destroy, style="Secondary.TButton").pack(side="right")
    ttk.Button(actions, text="新增到資料庫", command=insert_task, style="Success.TButton").pack(side="right", padx=(0, 10))

    entries[0].focus_set()


def insert_task():
    brand = B.get().strip()
    model = M.get().strip()

    if not brand or not model:
        msgbox.showwarning("資料不完整", "品牌與車型不能留白。")
        return

    values = (brand, model, P.get(), H.get(), T.get(), R.get().strip(), D.get(), Y.get())

    ans = msgbox.askokcancel(
        "確認新增",
        f"確定要新增以下資料嗎？\n\n品牌：{brand}\n車型：{model}\n價格：{P.get():,.0f} USD",
        icon="info",
    )
    if not ans:
        return

    try:
        conn = sqlite3.connect(DB_PATH)
        conn.execute("INSERT INTO supercars VALUES (?, ?, ?, ?, ?, ?, ?, ?)", values)
        conn.commit()
        conn.close()
        msgbox.showinfo("新增完成", f"{brand} {model} 已成功寫入資料庫。")
    except Exception as exc:
        show_db_error(exc)


# ============================================================
# 主畫面 Dashboard
# ============================================================
def dashboard_card(parent, number, title, description, button_text, command, accent):
    outer, card = make_card(parent, 20, 18)

    top = tk.Frame(card, bg=COLORS["surface"])
    top.pack(fill="x")

    badge = tk.Label(
        top,
        text=number,
        font=("Microsoft JhengHei", 9, "bold"),
        fg="white",
        bg=accent,
        width=4,
        pady=4,
    )
    badge.pack(side="left")

    make_label(card, title, 15, "bold").pack(anchor="w", pady=(16, 7))
    make_label(
        card,
        description,
        10,
        "normal",
        COLORS["muted"],
        COLORS["surface"],
        justify="left",
        wraplength=270,
    ).pack(anchor="w", fill="x")

    tk.Frame(card, bg=COLORS["surface"], height=14).pack()
    ttk.Button(card, text=button_text, command=command, style="Secondary.TButton").pack(anchor="w")
    return outer


def build_main_window():
    global win

    win = tk.Tk()
    win.title("Supercar Analytics｜超跑資料分析系統")
    win.configure(bg=COLORS["bg"])
    center_window(win, 1120, 760)
    win.minsize(1000, 700)

    setup_style()

    # Header
    header = tk.Frame(win, bg=COLORS["navy"], height=190)
    header.pack(fill="x")
    header.pack_propagate(False)

    hero = tk.Frame(header, bg=COLORS["navy"])
    hero.pack(fill="both", expand=True, padx=44, pady=30)

    left = tk.Frame(hero, bg=COLORS["navy"])
    left.pack(side="left", fill="both", expand=True)

    make_label(left, "SUPERCAR ANALYTICS", 10, "bold", "#93C5FD", COLORS["navy"]).pack(anchor="w")
    make_label(left, "超跑資料分析系統", 27, "bold", "white", COLORS["navy"]).pack(anchor="w", pady=(6, 4))
    make_label(
        left,
        "查詢車款、比較價格與性能、建立 3D 關係圖，並使用線性回歸預測車輛屬性。",
        11,
        "normal",
        "#CBD5E1",
        COLORS["navy"],
    ).pack(anchor="w")

    db_panel = tk.Frame(hero, bg="#1F2937", padx=18, pady=14)
    db_panel.pack(side="right", anchor="n")
    make_label(db_panel, "DATABASE", 9, "bold", "#94A3B8", "#1F2937").pack(anchor="w")
    make_label(db_panel, "car.db", 14, "bold", "white", "#1F2937").pack(anchor="w", pady=(3, 0))
    make_label(db_panel, DB_PATH, 8, "normal", "#CBD5E1", "#1F2937").pack(anchor="w", pady=(2, 0))

    # Body
    body = tk.Frame(win, bg=COLORS["bg"])
    body.pack(fill="both", expand=True, padx=38, pady=28)

    section_top = tk.Frame(body, bg=COLORS["bg"])
    section_top.pack(fill="x", pady=(0, 16))
    make_label(section_top, "分析工具", 17, "bold", bg=COLORS["bg"]).pack(side="left")
    make_label(section_top, "選擇一個功能開始分析", 10, "normal", COLORS["muted"], COLORS["bg"]).pack(side="left", padx=(12, 0), pady=(4, 0))
    ttk.Button(section_top, text="＋ 新增車輛資料", command=input_, style="Success.TButton").pack(side="right")

    grid = tk.Frame(body, bg=COLORS["bg"])
    grid.pack(fill="both", expand=True)
    for col in range(3):
        grid.grid_columnconfigure(col, weight=1, uniform="card")
    for row in range(2):
        grid.grid_rowconfigure(row, weight=1)

    cards = [
        ("01", "價格趨勢", "依品牌查看車款價格隨推出年份的變化。", "查看價格趨勢", money, COLORS["primary"]),
        ("02", "價格 × 外觀", "比較市場價格與外觀評分之間的關係。", "查看外觀關係", appearance, COLORS["violet"]),
        ("03", "價格 × 馬力", "比較超跑售價與馬力輸出的變化趨勢。", "查看馬力關係", horse, COLORS["orange"]),
        ("04", "3D 多屬性分析", "自由選擇三個屬性建立三維散佈關係圖。", "建立 3D 圖", datafor3D, COLORS["green"]),
        ("05", "品牌車型查詢", "從資料庫快速查詢指定品牌與車型完整規格。", "開始查詢", serch, "#0F766E"),
        ("06", "屬性趨勢預測", "以線性回歸分析品牌資料並預測另一項屬性。", "開啟預測工具", predict_menu, "#4F46E5"),
    ]

    for i, card_info in enumerate(cards):
        card = dashboard_card(grid, *card_info)
        card.grid(row=i // 3, column=i % 3, sticky="nsew", padx=8, pady=8)

    # Footer / Menu
    footer = tk.Frame(win, bg=COLORS["surface"], height=42)
    footer.pack(fill="x", side="bottom")
    footer.pack_propagate(False)
    make_label(
        footer,
        "SQLite + Pandas + Matplotlib｜資料庫位置可在程式最上方 DB_PATH 修改",
        9,
        "normal",
        COLORS["muted"],
        COLORS["surface"],
    ).pack(side="left", padx=28, pady=11)

    menu = tk.Menu(win)
    win.config(menu=menu)

    chart_menu = tk.Menu(menu, tearoff=False)
    menu.add_cascade(label="圖表分析", menu=chart_menu)
    chart_menu.add_command(label="價格趨勢", command=money)
    chart_menu.add_command(label="價格 × 外觀", command=appearance)
    chart_menu.add_command(label="價格 × 馬力", command=horse)
    chart_menu.add_command(label="3D 視覺化", command=datafor3D)

    data_menu = tk.Menu(menu, tearoff=False)
    menu.add_cascade(label="資料", menu=data_menu)
    data_menu.add_command(label="車輛查詢", command=serch)
    data_menu.add_command(label="新增車輛", command=input_)

    predict_menu_bar = tk.Menu(menu, tearoff=False)
    menu.add_cascade(label="預測", menu=predict_menu_bar)
    predict_menu_bar.add_command(label="屬性趨勢預測", command=predict_menu)

    menu.add_command(label="離開", command=win.destroy)

    return win


# ============================================================
# 啟動程式
# ============================================================
if __name__ == "__main__":
    app = build_main_window()
    app.mainloop()

In [2]:
from tkinter import ttk       
import tkinter as tk 
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
from tkinter import messagebox as msgbox
import sqlite3 as s
import sqlite3
import pandas as pd
from matplotlib.ticker import ScalarFormatter
plt.rcParams['font.family'] = ['Microsoft JhengHei']
plt.rcParams['axes.unicode_minus'] = False
#from tkinter import END #特別注意
from PIL import ImageTk,Image
col_map = {"車型":"brand",
           "型號":"model",
           "價格":"price_usd",
           "馬力":"horsepower",
           "最高時速":"top_speed",
           "稀有度":"rarity",
           "外觀":"design_rating",
           "推出年分":"release_year"}
def predict_menu():
    global combo_brand, combo_x, combo_y, p1, canvas_frame
    pr_win = tk.Toplevel()
    p1 = tk.DoubleVar()
    pr_win.title("車輛屬性預測圖")
    pr_win.geometry("700x650")
    
    tk.Label(pr_win, text="選擇品牌：").place(x=20, y=20)
    combo_brand = ttk.Combobox(pr_win, values=get_brands())
    combo_brand.place(x=100, y=20)
    if get_brands():
        combo_brand.set(get_brands()[0])
    
    # 屬性清單
    attrs = ["價格", "馬力", "最高時速", "外觀", "推出年分"]
    
    tk.Label(pr_win, text="X軸屬性：").place(x=20, y=60)
    combo_x = ttk.Combobox(pr_win, values=attrs)
    combo_x.place(x=80, y=60)
    combo_x.set("價格")
    
    tk.Label(pr_win, text="Y軸屬性：").place(x=250, y=60)
    combo_y = ttk.Combobox(pr_win, values=attrs)
    combo_y.place(x=310, y=60)
    combo_y.set("馬力")

    tk.Button(pr_win, text='生成預測圖', width='20', command=predict_car).place(x=500, y=55)
    
    # 輸入區域
    tk.Label(pr_win, text="輸入X軸數值預測Y軸數值：").place(x=60, y=580)
    tk.Entry(pr_win, width=10, textvariable=p1).place(x=220, y=580)
    tk.Button(pr_win, text='預測', width='20', command=start_car_predict).place(x=340, y=575)
    
    canvas_frame = tk.Frame(pr_win, width=660, height=380)
    canvas_frame.place(x=20, y=100)

def predict_car():
    global a, b, x_vals, y_vals, x_col, y_col
    brand = combo_brand.get()
    x_col = col_map[combo_x.get()]
    y_col = col_map[combo_y.get()]

    conn = sqlite3.connect(r"D:\database\car.db")
    df = pd.read_sql_query(
        f"SELECT {x_col}, {y_col} FROM supercars WHERE brand = ? ORDER BY {x_col}", conn, params=(brand,)
    )
    conn.close()

    if df.empty:
        msgbox.showwarning("警告", "此品牌沒有資料！")
        return

    x_vals = np.array(df[x_col])
    y_vals = np.array(df[y_col])
    x_mean = np.mean(x_vals)
    y_mean = np.mean(y_vals)
    numerator = np.sum((x_vals - x_mean) * (y_vals - y_mean))
    denominator = np.sum((x_vals - x_mean) ** 2)
    if denominator == 0:
        msgbox.showwarning("錯誤", "資料不足或變異為零，無法建立模型！")
        return
    a = numerator / denominator
    b = y_mean - a * x_mean

    # 畫圖
    fig = Figure(figsize=(6, 4), dpi=100)
    ax = fig.add_subplot(111)
    ax.scatter(x_vals, y_vals, color='blue', label='實際數據')
    ax.plot(x_vals, a * x_vals + b, color='red', label='線性回歸線')
    ax.ticklabel_format(style='plain', axis='y')  # 強制不用科學記號
    ax.ticklabel_format(style='plain', axis='x')
    ax.set_xlabel(combo_x.get())
    ax.set_ylabel(combo_y.get())
    if combo_x.get() == "價格":
        ax.set_xlabel("價格(usd)")
    elif combo_y.get() == "價格":
        ax.set_ylabel("價格(usd)")
    ax.set_title(f"{brand}：{combo_x.get()} 與 {combo_y.get()} 的線性關係")
    ax.legend()
    ax.grid(True)
    fig.tight_layout()

    # 更新顯示
    for widget in canvas_frame.winfo_children():
        widget.destroy()
    canvas = FigureCanvasTkAgg(fig, master=canvas_frame)
    canvas.draw()
    canvas.get_tk_widget().pack()

def start_car_predict():
    if not (combo_x.get() and combo_y.get()):
        msgbox.showwarning("警告", "請先選擇屬性並生成預測圖！")
        return
    
    if p1.get() == 0:
        msgbox.showwarning("警告", "請輸入數值！")
        return
    x_input = p1.get()
    y_pred = a * x_input + b
    msgbox.showinfo("預測結果", f"當 {combo_x.get()} = {x_input:.2f} 時，預測 {combo_y.get()} ≈ {y_pred:.2f}")
def get_brands():
    # 從資料庫抓取品牌列表
    conn = sqlite3.connect(r"D:\database\car.db")
    cursor = conn.cursor()
    cursor.execute("SELECT DISTINCT brand FROM supercars")
    brands = [row[0] for row in cursor.fetchall()]
    conn.close()
    return brands
def money():
    global combo1 , fig_win ,fig
    fig_win = tk.Toplevel()
    fig_win.title("價錢趨勢折線圖")
    fig_win.geometry("640x550")
    fig = Figure(figsize=(6, 4), dpi=100)
    pc = get_brands()
    combo1 = ttk.Combobox(fig_win, values=pc)
    combo1.place(x=10, y=40)
    combo1.set("Lamborghini")
    tk.Button(fig_win, text='生成折線圖', width='20', command=find).place(x=200, y=35)
def find():
    global list_1 , list_2 
    list_1 = []
    list_2 = []
    conn=s.connect(r"D:\database\car.db")#建立資料庫連線
    cursor = conn.cursor()
    cursor.execute("SELECT release_year , price_usd FROM supercars WHERE brand = ? order by release_year",(combo1.get(),))
    rows = cursor.fetchall()
    for  year, price  in rows :
        list_1.append(price)
        list_2.append(year)
    fig.clf()
    conn.close()
    ax = fig.add_subplot(111)
    ax.plot(list_2, list_1, 'g:*', label=f"{combo1.get()}的價錢趨勢")
    ax.set_title(combo1.get(), fontsize=24)
    ax.set_ylabel('price(usd)', fontsize=14)
    ax.set_xlabel('year', fontsize=14)
    ax.legend(loc='upper left')
    ax.ticklabel_format(style='plain', axis='y')  # 強制不用科學記號
    ax.grid(True)
    canvas = FigureCanvasTkAgg(fig, master=fig_win)
    fig.tight_layout()
    canvas.draw()
    canvas.get_tk_widget().place(x=10, y=80, width=620, height=450)
def appearance():
    global combo2 , win_1 ,fig
    win_1 = tk.Toplevel()
    win_1.title("價錢對於外觀的成長圖")
    win_1.geometry("640x550")
    fig = Figure(figsize=(6, 4), dpi=100)
    ac = get_brands()
    combo2 = ttk.Combobox(win_1, values=ac)
    combo2.place(x=10, y=40)
    combo2.set("Lamborghini")
    tk.Button(win_1, text='生成折線圖', width='20', command=lambda: find2("appearance"
                                                             ,"SELECT design_rating , price_usd FROM supercars WHERE brand = ? order by price_usd"
                                                             ,combo2.get()
                                                             ,win_1
                                                            )).place(x=200, y=35)

def horse():
    global combo3 , win_4 ,fig
    win_4 = tk.Toplevel()
    win_4.title("價錢對於馬力的關係圖")
    win_4.geometry("640x550")
    fig = Figure(figsize=(6, 4), dpi=100)
    ac = get_brands()
    combo3 = ttk.Combobox(win_4, values=ac)
    combo3.place(x=10, y=40)
    combo3.set("Lamborghini")
    tk.Button(win_4, text='生成折線圖', width='20', command=lambda: find2("horsepower"
                                                             ,"SELECT horsepower , price_usd FROM supercars WHERE brand = ? order by price_usd"
                                                             ,combo3.get()
                                                             ,win_4
                                                            )).place(x=200, y=35)
def find2(A ,code , C , win):
    global list__1 , list__2 
    list__1 = []
    list__2 = []
    conn=s.connect(r"D:\database\car.db")#建立資料庫連線
    cursor = conn.cursor()
    cursor.execute(code,(C,))
    rows = cursor.fetchall()
    for  a, price  in rows :
        list__1.append(price)
        list__2.append(a)
    fig.clf()
    conn.close()
    ax = fig.add_subplot(111)
    ax.plot(list__1, list__2, 'g:*', label=f"{C}的趨勢")
    ax.set_title(C, fontsize=24)
    ax.set_xlabel('price(usd)', fontsize=14)
    ax.set_ylabel(A, fontsize=14)
    ax.legend(loc='upper left')
    ax.ticklabel_format(style='plain', axis='y')  # 強制不用科學記號
    ax.ticklabel_format(style='plain', axis='x')
    ax.grid(True)
    canvas = FigureCanvasTkAgg(fig, master=win)
    fig.tight_layout()
    canvas.draw()
    canvas.get_tk_widget().place(x=10, y=80, width=620, height=450)
def datafor3D():
    global comboa , combob , comboc
    w_2 = tk.Toplevel()
    w_2.title("價值定義")
    w_2.geometry('550x200')
    w_2.config(bg='#ADD8E6')
    st = ["價格","馬力","最高時速","外觀","推出年分"]
    tk.Label(w_2, text='價值定義', font='14').place(x=10, y=10)
    comboa = ttk.Combobox(w_2, values=st)
    comboa.place(x=10, y=40)
    comboa.set("價格")
    combob = ttk.Combobox(w_2, values=st)
    combob.place(x=180, y=40)
    combob.set("馬力")
    comboc = ttk.Combobox(w_2, values=st)
    comboc.place(x=350, y=40)
    comboc.set("外觀")
    tk.Button(w_2, text='確定', width='20', command=bc3D).place(x=180, y=100)
def bc3D():
    
    x_col = comboa.get()
    y_col = combob.get()
    z_col = comboc.get()

    conn = s.connect(r"D:\database\car.db")
    df = pd.read_sql_query(
        f"SELECT {col_map[x_col]}, {col_map[y_col]}, {col_map[z_col]} FROM supercars", conn
    )
    conn.close()

    X = df[col_map[x_col]].values
    Y = df[col_map[y_col]].values
    Z = df[col_map[z_col]].values
    w_3 = tk.Toplevel()
    w_3.title('3D視覺化')
    w_3.geometry('800x700')
    w_3.config(bg='#ADD8E6')
    
    fig = Figure(figsize=(10, 8), dpi=100)
    ax = fig.add_subplot(111, projection='3d')
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis.set_major_formatter(ScalarFormatter())
        axis.get_major_formatter().set_scientific(False)
    value = np.random.uniform(0, 100, len(X))
    color_map = plt.colormaps['prism']
    scatter = ax.scatter(X,Y,Z, c=value, cmap=color_map, alpha=0.7, s=20)  
    ax.set_xlabel(x_col, fontsize=12)
    ax.set_ylabel(y_col, fontsize=12)
    ax.set_zlabel(z_col, fontsize=12)
    ax.set_title(f'{x_col}-{y_col}-{z_col} 的 3D 關係', fontsize=14)
    canvas = FigureCanvasTkAgg(fig, master=w_3)
    canvas.draw()
    canvas.get_tk_widget().pack(side=tk.TOP, fill=tk.BOTH, expand=1)
def serch():
    global brand_combo ,model_combo , tree
    win_5 = tk.Toplevel()
    win_5.title("品牌車型查詢")
    win_5.geometry("1250x120")

    tk.Label(win_5, text="選品牌").place(x=20,y=10)
    brands = get_brands()  # 從資料庫抓品牌
    brand_combo = ttk.Combobox(win_5, values=brands)
    brand_combo.place(x=70, y=10)
    
  
    if brands:
        brand_combo.set(brands[0])
    brand_combo.bind("<<ComboboxSelected>>", update_models)

    tk.Label(win_5, text="選車型").place(x=250,y=10)
    model_combo = ttk.Combobox(win_5, values=[])
    model_combo.place(x=300 , y=10)

    tk.Button(win_5, text="查詢資料", command=show_car_info).place(x=500 , y=8)

    # 建立 Treeview
    columns = ("品牌","車型","價格","馬力","最高時速","稀有度","外觀","推出年分")
    tree = ttk.Treeview(win_5, columns=columns, show='headings', height=1)
    for col in columns:
        tree.heading(col, text=col)
        tree.column(col, width=150, anchor='center')
    tree.place(x=10,y=60)

    win_5.mainloop()
def update_models(event):
    global brand_combo , model_combo
    brand = brand_combo.get()
    conn = sqlite3.connect(r"D:\database\car.db")
    cursor = conn.cursor()
    cursor.execute("SELECT model FROM supercars WHERE brand = ?", (brand,))
    models = [row[0] for row in cursor.fetchall()]
    conn.close()
    
    model_combo['values'] = models
    if models:
        model_combo.set(models[0])
    else:
        model_combo.set('')
def show_car_info():
    # 清空 Treeview
    for i in tree.get_children():
        tree.delete(i)
    
    brand = brand_combo.get()
    model = model_combo.get()
    conn = sqlite3.connect(r"D:\database\car.db")
    cursor = conn.cursor()
    cursor.execute("SELECT brand, model, price_usd, horsepower, top_speed, rarity, design_rating, release_year FROM supercars WHERE brand = ? AND model = ?", (brand, model))
    car_data = cursor.fetchone()
    conn.close()
    
    if car_data:
        tree.insert('', 'end', values=car_data)
def input_():
    global B , M , P , H , T , R , D , Y 
    w_5=tk.Toplevel()                                                      
    w_5.title('新增功能')
    w_5.geometry("850x100")
    w_5.config(bg='#00E3E3')
    B=tk.StringVar() 
    M=tk.StringVar()
    P=tk.DoubleVar()
    H=tk.IntVar()
    T=tk.DoubleVar()
    R=tk.StringVar()
    D=tk.DoubleVar()
    Y=tk.IntVar()
    tk.Label(w_5, text='品牌',font='12',bg='#00E3E3').place(x=10,y=10)    
    tk.Entry(w_5, width=10, textvariable=B).place(x=10,y=50)
    tk.Label(w_5, text='車型',font='12',bg='#00E3E3').place(x=110,y=10)
    tk.Entry(w_5, width=10, textvariable=M).place(x=110,y=50)
    tk.Label(w_5, text='價格',font='12',bg='#00E3E3').place(x=210,y=10)
    tk.Entry(w_5, width=10, textvariable=P).place(x=210,y=50)
    tk.Label(w_5, text='馬力',font='12',bg='#00E3E3').place(x=310,y=10)
    tk.Entry(w_5, width=10, textvariable=H).place(x=310,y=50)
    tk.Label(w_5, text='極限速度',font='12',bg='#00E3E3').place(x=410,y=10)
    tk.Entry(w_5, width=10, textvariable=T).place(x=410,y=50)
    tk.Label(w_5, text='稀有度',font='12',bg='#00E3E3').place(x=510,y=10)
    tk.Entry(w_5, width=10, textvariable=R).place(x=510,y=50)
    tk.Label(w_5, text='外觀評分',font='12',bg='#00E3E3').place(x=610,y=10)
    tk.Entry(w_5, width=10, textvariable=D).place(x=610,y=50)
    tk.Label(w_5, text='推出年分',font='12',bg='#00E3E3').place(x=710,y=10)
    tk.Entry(w_5, width=10, textvariable=Y).place(x=710,y=50)
    tk.Button(w_5,text ='新增', command=insert_task).place(x=800,y=48)
def insert_task():
    B_=B.get()
    M_=M.get()
    P_=P.get()
    H_=H.get()
    T_=T.get()
    R_=R.get()
    D_=D.get()
    Y_=Y.get()
    ans=msgbox.askokcancel('資料新增','確定寫入資料',icon='info')
    if ans==True:
        conn=s.connect(r"D:\database\car.db")#建立資料庫連線
        x=(B_,M_,P_,H_,T_,R_,D_,Y_)#tuple 
        sql='insert into supercars values(?,?,?,?,?,?,?,?)'
        conn.execute(sql,x)
        conn.commit()#新增、刪除、修改，查詢不用
        conn.close()
win = tk.Tk()
win.title("車價買到的東西成長圖")
win.geometry('400x150')
win.config(bg='#A9A9A9')
text_label = tk.Label(
    win,
    text="車子價錢影響的趨勢",
    font=("Arial", 30),
    bg="#FFFFFF",
    fg="black",
    justify="center"
)
text_label.place(x=10, y=40)
menu = tk.Menu(win)
win.config(menu=menu)
simu_menu = tk.Menu(menu, tearoff=False)
menu.add_cascade(label="生成圖表", menu=simu_menu)
simu_menu.add_command(label="價錢成長折線圖", command=money)
simu_menu.add_command(label="外觀漲幅和金錢關係折線圖", command=appearance)
simu_menu.add_command(label="馬力和金錢關係折線圖", command=horse)
simu_menu.add_command(label="3D視覺化", command=datafor3D)
simu_menu.add_command(label="結束", command=win.destroy)

kimu_menu = tk.Menu(menu, tearoff=False)
menu.add_cascade(label="車輛查詢", menu=kimu_menu)
kimu_menu.add_command(label="查詢車輛", command=serch)
iimu_menu = tk.Menu(menu, tearoff=False)

pimu_menu = tk.Menu(menu, tearoff=False)
menu.add_cascade(label="屬性趨勢預測圖", menu=pimu_menu)
pimu_menu.add_command(label="屬性趨勢預測圖", command=predict_menu)

menu.add_cascade(label="資料輸入", menu=iimu_menu)
iimu_menu.add_command(label="資料輸入", command=input_)
win.mainloop()
